In [1]:
import re
import pandas as pd
import sqlite3
from great_tables import gt

In [2]:
conn=sqlite3.connect('Data/wnba_database.db')
current_wnba_season=2026

In [3]:
standings=(
    pd.read_sql_query('SELECT * FROM standings WHERE season = ?',
    con=conn,
    params=[current_wnba_season])
)

In [4]:
player_gp_percentages=(
    pd.read_sql_query('SELECT * FROM player_bios WHERE season = ?',
    con=conn,
    params=[current_wnba_season])
).merge(
    standings.filter(regex="Team"),
        left_on="TEAM_ID",
        right_on="TeamID",
        how="left"
).assign(G_PERCENT=lambda d: d.GP / d.TeamGP)


In [5]:
player_totals_w_gp_percentages=(
    pd.read_sql_query('SELECT * FROM player_totals WHERE season = ?',
    con=conn,
    params=[current_wnba_season])
).merge(player_gp_percentages,how='left')

## The Deadshot Award (presented by Sue Bird)

best qualifying 3 point percentage, minimum 30 3PM

In [6]:
gt.GT(
    player_totals_w_gp_percentages.query('FG3M>=30').nlargest(5,'FG3_PCT')
    .assign(FG3M_per_game=lambda d: d['FG3M']/d['GP'])
    .loc[:, ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'FG3M','FG3M_per_game', 'FG3_PCT']]  # select columns
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(columns=["FG3_PCT"]).fmt_number(columns=['FG3M_per_game'],decimals=2)

player,team,FG3M,FG3M_per_game,FG3_PCT
Kelsey Mitchell,IND,135,3.07,45.00%
Maddy Siegrist,DAL,35,0.85,42.20%
Jovana Nogic,PHX,32,2.00,42.10%
Julie Allemand,TOR,39,1.18,41.90%
Sophie Cunningham,IND,67,1.56,41.90%


## The Stormtrooper Award

worst qualifying 2 point percentage, minimum 55 2PM

In [7]:
gt.GT(
    player_totals_w_gp_percentages
    .assign(
      FG2M=lambda d: d['FGM']-d['FG3M'],
      FG2A=lambda d: d['FGA']-d['FG3A']
    )
    .assign(FG2_PCT=lambda d: d['FG2M']/d['FG2A'])
    .query('FG2M>=55').nsmallest(5,'FG2_PCT')
    .assign(FG2M_per_game=lambda d: d['FG2M']/d['GP'])
    .loc[:, ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'FG2M','FG2M_per_game', 'FG2_PCT']]  # select columns
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(columns=["FG2_PCT"]).fmt_number(columns=['FG2M_per_game'],decimals=2)

player,team,FG2M,FG2M_per_game,FG2_PCT
Zia Cooke,SEA,67,1.63,38.07%
Diamond Miller,CON,85,1.98,39.53%
Skylar Diggins,CHI,61,3.21,39.61%
Saniya Rivers,CON,90,2.37,40.18%
Charlisse Leger-Walker,CON,70,1.59,41.42%


## The Most Ethical Scorer Award (sponsored by Immanuel Kant)

lowest percentage of points from free throws, minimum 10 PPG and 70% of games played (credit to adc1369 & yeahright17 for the idea)

In [8]:
ethical_scoring=(
    player_totals_w_gp_percentages
    .assign(
        FT_PERCENT_OF_PTS=lambda d: d['FTM']/d['PTS'],
        FT_per_game=lambda d: d['FTM']/d['GP'],
        PTS_per_game=lambda d: d['PTS']/d['GP']
        )
        .loc[:,['PLAYER_NAME', 'TEAM_ABBREVIATION','GP','G_PERCENT', 'FT_per_game','PTS_per_game', 'FT_PERCENT_OF_PTS']]
)

In [9]:
gt.GT(
    ethical_scoring
    .query('PTS_per_game>=10 and G_PERCENT>=0.7')
    .nsmallest(5,'FT_PERCENT_OF_PTS')
    .loc[:,['PLAYER_NAME', 'TEAM_ABBREVIATION', 'FT_per_game','PTS_per_game', 'FT_PERCENT_OF_PTS']]
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(columns=["FT_PERCENT_OF_PTS"]).fmt_number(columns=['FT_per_game','PTS_per_game'],decimals=2)

player,team,FT_per_game,PTS_per_game,FT_PERCENT_OF_PTS
Courtney Williams,MIN,1.00,13.77,7.26%
Awa Fam,SEA,0.86,10.27,8.42%
Megan DiLeo,PDX,1.61,13.07,12.31%
Emily Engstler,PDX,1.27,10.14,12.56%
Nneka Ogwumike,LAS,2.19,17.26,12.67%


## The Least Ethical Scorer Award (sponsored by ~~Enron~~ ~~Aspiration~~ no one)

highest percentage of points from free throws, minimum 10 PPG and 70% of games played (credit to Banichi-aiji & yeahright17 for the idea)

In [10]:
gt.GT(
    ethical_scoring
    .query('PTS_per_game>=10 and G_PERCENT>=0.7')
    .nlargest(5,'FT_PERCENT_OF_PTS')
    .loc[:,['PLAYER_NAME', 'TEAM_ABBREVIATION', 'FT_per_game','PTS_per_game', 'FT_PERCENT_OF_PTS']]
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(columns=["FT_PERCENT_OF_PTS"]).fmt_number(columns=['FT_per_game','PTS_per_game'],decimals=2)

player,team,FT_per_game,PTS_per_game,FT_PERCENT_OF_PTS
Carla Leite,PDX,5.51,16.05,34.35%
Breanna Stewart,NYL,5.81,20.83,27.89%
Sonia Citron,WAS,4.65,16.77,27.72%
Angel Reese,ATL,4.51,16.40,27.52%
Jordin Canada,ATL,3.09,11.60,26.65%


## The "If She Dies, She Dies" Award (presented by Tom Thibodeau, sponsored by Ivan Drago)

most minutes played per game, minimum 70% of games played (credit to FurryCrew for the idea)

In [11]:
gt.GT(
    player_totals_w_gp_percentages
    .query('G_PERCENT>=0.7')
    .assign(MIN_PER_GAME=lambda d: d['MIN']/d['GP'])
    .nlargest(5,'MIN_PER_GAME')
    .loc[:,['PLAYER_NAME', 'TEAM_ABBREVIATION', 'GP','MIN_PER_GAME']]
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_number('MIN_PER_GAME')

player,team,GP,MIN_PER_GAME
Rhyne Howard,ATL,43,34.23
Alyssa Thomas,PHX,42,33.16
Sonia Citron,WAS,40,32.99
Breanna Stewart,NYL,42,32.95
Kahleah Copper,PHX,40,32.92


alternatively: most total minutes played (credit to FrankEMartindale for the idea)

In [12]:
gt.GT(
    player_totals_w_gp_percentages
    .nlargest(5,'MIN')
    .loc[:,['PLAYER_NAME', 'TEAM_ABBREVIATION', 'GP','MIN']]
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_number('MIN')

player,team,GP,MIN
Rhyne Howard,ATL,43,"1,471.76"
Allisha Gray,ATL,44,"1,433.65"
Kelsey Mitchell,IND,44,"1,430.36"
Kayla McBride,MIN,44,"1,406.28"
Chelsea Gray,LVA,43,"1,406.03"


## The "Black Hole" Award

most FGAs per assist, minimum 50% of games played (credit to Moose4KU for the idea)

In [13]:
fga_per_ast=(
    player_totals_w_gp_percentages
    .assign(FGA_PER_AST=lambda d: d['FGA']/d['AST'])
    .loc[:,['PLAYER_NAME','TEAM_ABBREVIATION','GP','G_PERCENT','MIN','FGA','AST','FGA_PER_AST']]
)

In [14]:
gt.GT(
    fga_per_ast
    .query('G_PERCENT>=0.5')
    .nlargest(5,'FGA_PER_AST',keep='all')
    .sort_values(['FGA_PER_AST','FGA'],ascending=False)
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_number(['MIN','FGA_PER_AST']).fmt_percent('G_PERCENT')

player,team,GP,G_PERCENT,MIN,FGA,AST,FGA_PER_AST
Madina Okot,ATL,42,95.45%,381.77,157,11,14.27
Megan DiLeo,PDX,41,93.18%,994.28,380,37,10.27
Kayla Thornton,GSV,44,100.00%,"1,069.62",302,30,10.07
Sika Kone,ATL,35,79.55%,174.03,61,7,8.71
Kahleah Copper,PHX,40,90.91%,"1,316.80",643,75,8.57


## The "Hot Potato" Award

fewest FGAs per assist, minimum 50% of games played (credit to Moose4KU for the idea & ajayod for the name)

In [15]:
gt.GT(
    fga_per_ast
    .query('G_PERCENT>=0.5')
    .nsmallest(5,'FGA_PER_AST')
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_number(['MIN','FGA_PER_AST']).fmt_percent('G_PERCENT')

player,team,GP,G_PERCENT,MIN,FGA,AST,FGA_PER_AST
Julie Allemand,TOR,33,75.00%,933.38,141,180,0.78
Jordin Canada,ATL,43,97.73%,"1,338.34",378,313,1.21
Alyssa Thomas,PHX,42,95.45%,"1,392.92",443,358,1.24
Courtney Vandersloot,CHI,27,61.36%,605.25,198,147,1.35
Sug Sutton,DAL,25,56.82%,257.37,74,50,1.48


## The "Most Expendable Player" Award (sponsored by the National Basketball Referees Association, presented by Sylvester Stallone)

highest personal fouls per 30 minutes, minimum 50% of games played and 10 MPG (credit to PsychoM & BrightGreenLED for the idea)

In [16]:
gt.GT(
    player_totals_w_gp_percentages
    .assign(MIN_PER_GAME=lambda d: d['MIN']/d['GP'])
    .query('G_PERCENT>=0.5 and MIN_PER_GAME>=10')
    .assign(PF_PER_30_MINS=lambda d: d['PF']/d['MIN']*30)
    .loc[:,['PLAYER_NAME','TEAM_ABBREVIATION','MIN','PF','PF_PER_30_MINS']]
    .nlargest(5,'PF_PER_30_MINS')
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_number(['MIN','PF_PER_30_MINS'])

player,team,MIN,PF,PF_PER_30_MINS
Raegan Beers,CON,478.75,108,6.77
Cameron Brink,LAS,594.96,111,5.60
Cheyenne Parker-Tyus,LVA,348.40,57,4.91
Alysha Clark,DAL,341.90,55,4.83
Nell Angloma,CON,503.57,75,4.47


## The "[This Game Has Always Been, And Will Always Be, About Buckets](https://www.youtube.com/watch?v=-xYejfYxT4s)" Award

highest points as percentage of counting stats (rebounds, assists, steals, blocks), minimum of 70% of games played

In [17]:
pts_as_percent_counting_stats=(
    player_totals_w_gp_percentages
    .assign(
        PTS_AS_PERCENT_OF_OTHER_STATS=lambda d: d['PTS']/(d['PTS']+d['REB']+d['AST']+d['STL']+d['BLK']),
        PPG=lambda d: d['PTS']/d['GP'],
        RPG=lambda d: d['REB']/d['GP'],
        APG=lambda d: d['AST']/d['GP'],
        SPG=lambda d: d['STL']/d['GP'],
        BPG=lambda d: d['BLK']/d['GP']
        )
    .loc[:,['PLAYER_NAME','TEAM_ABBREVIATION','GP','G_PERCENT','PPG','RPG','APG','SPG','BPG','PTS_AS_PERCENT_OF_OTHER_STATS']]
)

In [18]:
gt.GT(
    pts_as_percent_counting_stats.
    query('G_PERCENT>=0.7')
    .nlargest(5,'PTS_AS_PERCENT_OF_OTHER_STATS')
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_number(['PPG','RPG','APG','SPG','BPG']).fmt_percent(['G_PERCENT','PTS_AS_PERCENT_OF_OTHER_STATS'])

player,team,GP,G_PERCENT,PPG,RPG,APG,SPG,BPG,PTS_AS_PERCENT_OF_OTHER_STATS
Kelsey Mitchell,IND,44,100.00%,24.68,1.70,2.75,1.05,0.11,81.47%
Kahleah Copper,PHX,40,90.91%,21.48,3.67,1.88,0.82,0.10,76.83%
Sydney Taylor,CHI,37,84.09%,13.62,1.70,1.97,0.68,0.24,74.78%
Marina Mabrey,TOR,32,72.73%,20.84,3.28,3.69,0.91,0.38,71.64%
Kayla McBride,MIN,44,100.00%,17.61,3.00,2.18,1.64,0.23,71.43%


## The "Imma Make 'Em Both" Award (presented by Diar DeRozan)*

biggest decline from non-clutch FT% to clutch FT%, min 15 clutch FTA (credit to Necessary_Career_253 for the idea and midnightgreen29 for the name)

In [19]:
clutch_and_totals=player_totals_w_gp_percentages.merge(
    pd.read_sql_query(
        'SELECT * FROM player_clutch WHERE season = ?',
        con=conn,
        params=[current_wnba_season]),
        how='left'
)

ft_div_by_clutch=(
    clutch_and_totals[['PLAYER_NAME','FTM','FTA','FTM_clutch','FTA_clutch']]
    .assign(
        FTM_non_clutch=lambda x: x.FTM-x.FTM_clutch,
        FTA_non_clutch=lambda x: x.FTA-x.FTA_clutch,
        FT_PCT_clutch=lambda x: x.FTM_clutch/x.FTA_clutch,
        FT_PCT_non_clutch=lambda x: x.FTM_non_clutch/x.FTA_non_clutch,
        differential=lambda x: x.FT_PCT_clutch-x.FT_PCT_non_clutch
    )
)

In [20]:
gt.GT(
    ft_div_by_clutch
    .query("FTA_clutch >= 15")
    .nsmallest(5, "differential",keep="all")
    [["PLAYER_NAME","FTA_clutch","FT_PCT_clutch","FT_PCT_non_clutch","differential"]]
).fmt_percent(
    columns=["FT_PCT_clutch","FT_PCT_non_clutch","differential"]
)

PLAYER_NAME,FTA_clutch,FT_PCT_clutch,FT_PCT_non_clutch,differential
Caitlin Clark,20.0,60.00%,87.39%,−27.39%
Paige Bueckers,18.0,72.22%,89.15%,−16.93%
Kelsey Mitchell,20.0,70.00%,86.63%,−16.63%
Olivia Miles,27.0,74.07%,89.16%,−15.08%
Gabby Williams,24.0,66.67%,75.26%,−8.59%


## The "Ice, Ice, Baby" Award (presented by Vanilla Ice)*

biggest improvement from non-clutch FT% to clutch FT%, min 15 clutch FTA

In [21]:
gt.GT(
    ft_div_by_clutch
    .query("FTA_clutch >= 15")
    .nlargest(5, "differential",keep="all")
    [["PLAYER_NAME","FTA_clutch","FT_PCT_clutch","FT_PCT_non_clutch","differential"]]
).fmt_percent(
    columns=["FT_PCT_clutch","FT_PCT_non_clutch","differential"]
)

PLAYER_NAME,FTA_clutch,FT_PCT_clutch,FT_PCT_non_clutch,differential
Allisha Gray,24.0,95.83%,79.36%,16.48%
Natasha Cloud,18.0,94.44%,83.84%,10.61%
Kiki Iriafen,24.0,70.83%,64.24%,6.59%
A'ja Wilson,25.0,92.00%,85.91%,6.09%
Jessica Shepard,22.0,72.73%,66.90%,5.83%


## The "Paint Allergy" Award

players w/highest % of shots from outside paint, min 150 FGA & 50% of games played (credit to frosiano for the original idea of highest percentage of 3FGA of total FGA, and to Drummallumin for the revised idea of all shots outside of 15 feet)

In [22]:
non_paint_fga=(
    pd.read_sql_query(
        'SELECT * FROM player_shot_locations WHERE season = ?',
        con=conn,
        params=[current_wnba_season])
    # compute totals and derived columns
    .pipe(lambda d: d.assign(
        tot_fga=lambda d: d.filter(like="_FGA").sum(axis=1),
        tot_fgm=lambda d: d.filter(like="_FGM").sum(axis=1)
    ))
    .pipe(lambda d: d.assign(
        non_paint_fga=d["tot_fga"] - d["Restricted_Area_FGA"] - d["In_The_Paint_Non_RA_FGA"],
        non_paint_fga_percentage=lambda d: d["non_paint_fga"] / d["tot_fga"]
    ))
    .rename(columns={
        '_PLAYER_ID': 'PLAYER_ID',
    })
)

In [23]:
gt.GT(
    non_paint_fga
    .merge(player_totals_w_gp_percentages, how='left')
    .loc[lambda df: (df['tot_fga'] >= 150) & (df['G_PERCENT'] >= 0.5)]
    .nlargest(5, 'non_paint_fga_percentage')
    .loc[:, ['PLAYER_NAME','TEAM_ABBREVIATION','tot_fga','non_paint_fga','non_paint_fga_percentage']]
    .rename(columns={'PLAYER_NAME': 'player','TEAM_ABBREVIATION': 'team'})
).fmt_percent(
    columns=["non_paint_fga_percentage"]
    )

player,team,tot_fga,non_paint_fga,non_paint_fga_percentage
Rachel Banham,CHI,192.0,167.0,86.98%
Marine Johannes,NYL,283.0,241.0,85.16%
Kia Nurse,TOR,207.0,171.0,82.61%
Jewell Loyd,LVA,304.0,249.0,81.91%
Janelle Salaun,GSV,388.0,302.0,77.84%


## The "Lumberjill" Award

players w/highest % of shots from inside paint, max height of 5'10" & min 150 FGA & minimum 50% of games played (credit to Drummallumin for the idea)

In [24]:
gt.GT(
    non_paint_fga.merge(player_totals_w_gp_percentages, how='left')
    # filter
    .loc[lambda d: (
        (d['tot_fga'] >= 150) &
        (d['G_PERCENT'] >= 0.5) &
        (d['PLAYER_HEIGHT_INCHES'] <= 5*12+10)
    )]
    # mutate
    .assign(
        paint_fga=lambda d: d['tot_fga'] - d['non_paint_fga'],
        paint_fga_percentage=lambda d: 1 - d['non_paint_fga_percentage']
    )
    # slice_max
    .nlargest(5, 'paint_fga_percentage')
    # select + rename
    .loc[:, ['PLAYER_NAME','TEAM_ABBREVIATION','PLAYER_HEIGHT','tot_fga','paint_fga','paint_fga_percentage']]
    .rename(columns={
        'PLAYER_NAME': 'player',
        'TEAM_ABBREVIATION': 'team'
    })
).fmt_percent(
    columns=["paint_fga_percentage"]
    )

player,team,PLAYER_HEIGHT,tot_fga,paint_fga,paint_fga_percentage
Carla Leite,PDX,5-9,407.0,267.0,65.60%
Jordin Canada,ATL,5-6,378.0,232.0,61.38%
Olivia Miles,MIN,5-10,567.0,345.0,60.85%
Kaitlyn Chen,GSV,5-9,241.0,145.0,60.17%
Tiffany Hayes,GSV,5-10,274.0,163.0,59.49%


## The "Fine, I'll Do It Myself" Award (sponsored by Thanos)

Highest percentage of unassisted field goals, minimum 50% of games played and 2 FGM per game

In [25]:
scoring_w_gp_qualify=(
    pd.read_sql_query(
        'SELECT * FROM player_scoring WHERE season = ?',
        con=conn,
        params=[current_wnba_season])
    .merge(
        player_totals_w_gp_percentages[['PLAYER_ID','G_PERCENT']],
        how='left'
    )
    .query('G_PERCENT>=0.5')
)

In [26]:
gt.GT(
    scoring_w_gp_qualify
    .query('FGM/GP>=2')    
    .nlargest(5, 'PCT_UAST_FGM')  # equivalent to slice_max
    .loc[:, ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'GP', 'MIN', 'FGM', 'PCT_UAST_FGM']]  # select columns
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(
    columns=["PCT_UAST_FGM"]
    )

player,team,GP,MIN,FGM,PCT_UAST_FGM
Olivia Miles,MIN,40,31.0,279,77.80%
Carla Leite,PDX,39,26.0,187,72.70%
Caitlin Clark,IND,40,31.1,282,68.80%
Teja Oblak,PDX,29,13.4,63,65.10%
Jordin Canada,ATL,43,31.1,163,62.00%


## The "You Gotta Feed Me" Award (presented by Sonya Thomas)

Highest percentage of assisted field goals, minimum 50% of games played and 1 FGM per game

In [27]:
gt.GT(
    scoring_w_gp_qualify
    .query('FGM/GP>=1')
    .nlargest(5, 'PCT_AST_FGM',keep='all')  # equivalent to slice_max
    .sort_values(['PCT_AST_FGM','FGM'],ascending=False)
    .loc[:, ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'GP', 'MIN', 'FGM', 'PCT_AST_FGM']]  # select columns
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(
    columns=["PCT_AST_FGM"]
    )

player,team,GP,MIN,FGM,PCT_AST_FGM
Alanna Smith,DAL,32,17.9,75,92.00%
Nia Coffey,MIN,40,23.0,97,91.80%
Awak Kuier,DAL,41,18.6,96,91.70%
Stefanie Dolson,SEA,44,15.0,46,91.30%
Naz Hillmon,ATL,43,28.7,147,90.50%
Kiah Stokes,GSV,43,22.3,63,90.50%
Antonia Delaere,MIN,39,11.7,42,90.50%


# ROUGH WORK